In [ ]:
import base64,hashlib,io,pathlib,subprocess,sys,zipfile
payload=base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest()=='e669da90c8b223552d8037b9b1a073baee50d5c07d98c2b09a612a350800a8b8'
root=pathlib.Path("/kaggle/working/t3d-project")
root.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    archive.extractall(root)


In [ ]:
import torch,json,shutil
assert torch.cuda.is_available(), "GPU required"
data=root/"artifacts/bpt-training-pilot"
manifest=json.loads((data/"manifest.json").read_text())
for asset in manifest["assets"]:
    for sample in asset["samples"]:
        matches=list(pathlib.Path("/kaggle/input").rglob(sample["file"]))
        assert len(matches)==1, sample["file"]
        shutil.copyfile(matches[0],data/sample["file"])
result=subprocess.run([sys.executable,"-m","unittest","discover","-s","tests","-p","test_bpt_ar.py","-v"],cwd=root,capture_output=True,text=True)
(root.parent/"bpt-ar-tests.log").write_text(result.stdout+result.stderr)
print(result.stdout+result.stderr)
assert result.returncode==0


In [ ]:
subprocess.run([sys.executable,"-m","tools.train_bpt_multiasset","--source-sha256",'e669da90c8b223552d8037b9b1a073baee50d5c07d98c2b09a612a350800a8b8'],cwd=root,check=True)
